<a href="https://colab.research.google.com/github/kunal399/dkai-tech-assessment/blob/main/task2/task2_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Task 2: ML Basics & Optimization using PySpark MLlib
Dataset      : Titanic (891 rows)
Baseline     : Random Forest Classifier
Improvement 1: Feature Engineering (Title + family features)
Improvement 2: Hyperparameter Tuning via CrossValidator

EVALUATION STRATEGY
───────────────────
Titanic has only 891 rows. A random 80/20 split gives ~145 test rows where
3 wrong predictions = 2% accuracy swing — making single-split scores
meaningless and unstable across runs.

Fix: Use 5-fold CV on the FULL dataset for all comparisons.
Each fold: train=713 rows, test=178 rows, averaged over 5 runs.
This gives stable, reproducible scores that actually reflect model quality.
"""

import warnings
warnings.filterwarnings("ignore")

import json
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, regexp_extract
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, Imputer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ─────────────────────────────────────────────────────────────
# 1. SPARK SESSION
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("TASK 2: ML Basics & Optimization — PySpark + Titanic")
print("=" * 60)

spark = SparkSession.builder \
    .appName("TitanicML") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"\n✅ Spark session started  (version {spark.version})")

# ─────────────────────────────────────────────────────────────
# 2. LOAD DATASET
# ─────────────────────────────────────────────────────────────
print("\n[1/6] Loading Titanic dataset...")

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
r   = requests.get(url)
with open("titanic.csv", "wb") as f:
    f.write(r.content)

df = spark.read.csv("titanic.csv", header=True, inferSchema=True)
df = df.select("Survived", "Pclass", "Sex", "Age", "Name",
               "SibSp", "Parch", "Fare", "Embarked")
df = df.fillna({"Embarked": "S"})
df.cache()

total    = df.count()
survived = df.filter(col("Survived") == 1).count()
print(f"   Rows       : {total}")
print(f"   Survived   : {survived} ({survived/total:.1%})")
print(f"   Evaluation : 5-fold CV on full dataset (713 train / 178 test per fold)")

# ─────────────────────────────────────────────────────────────
# 3. SHARED PREPROCESSING STAGES
# ─────────────────────────────────────────────────────────────
print("\n[2/6] Preprocessing stages ready...")

sex_indexer      = StringIndexer(inputCol="Sex",      outputCol="SexIdx",      handleInvalid="keep")
embarked_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIdx", handleInvalid="keep")
imputer_base     = Imputer(inputCols=["Age","Fare"],   outputCols=["Age","Fare"], strategy="median")

BASE_FEATURES  = ["Pclass", "SexIdx", "Age", "SibSp", "Parch", "Fare", "EmbarkedIdx"]
assembler_base = VectorAssembler(inputCols=BASE_FEATURES, outputCol="features", handleInvalid="keep")

# Evaluators
auc_eval = BinaryClassificationEvaluator(labelCol="Survived", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="Survived", predictionCol="prediction", metricName="f1")

def run_cv(pipeline, data, param_grid=None, nfolds=5):
    """
    Run cross-validation and return (best_model, avg_auc, avg_acc, avg_f1).
    Uses AUC as the CV selection metric, then re-evaluates acc+f1 on each fold.
    """
    if param_grid is None:
        param_grid = ParamGridBuilder().build()  # single combination

    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=param_grid,
        evaluator=auc_eval,
        numFolds=nfolds,
        seed=42,
        collectSubModels=False,
    )
    cv_model  = cv.fit(data)
    avg_auc   = max(cv_model.avgMetrics)

    # Refit best pipeline on full data to get train acc/f1
    best_pipeline = cv_model.bestModel
    preds = best_pipeline.transform(data)
    train_acc = acc_eval.evaluate(preds)
    train_f1  = f1_eval.evaluate(preds)

    return cv_model.bestModel, avg_auc, train_acc, train_f1

# ─────────────────────────────────────────────────────────────
# 4. BASELINE MODEL
# ─────────────────────────────────────────────────────────────
print("\n[3/6] Training Baseline Model  (5-fold CV)...")

rf_base = RandomForestClassifier(
    labelCol="Survived", featuresCol="features",
    numTrees=100, maxDepth=5, seed=42
)

pipeline_base = Pipeline(stages=[
    sex_indexer, embarked_indexer, imputer_base, assembler_base, rf_base
])

base_model, base_auc, base_train_acc, base_train_f1 = run_cv(pipeline_base, df)

print(f"   CV AUC-ROC  (avg 5 folds) : {base_auc:.4f}   ← primary metric")
print(f"   Train Accuracy (full fit) : {base_train_acc:.4f}")
print(f"   Train F1      (full fit)  : {base_train_f1:.4f}")

# ─────────────────────────────────────────────────────────────
# 5. IMPROVEMENT 1 — FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("\n[4/6] Improvement 1 — Feature Engineering  (5-fold CV)...")

def add_features(df):
    # Title: single column encoding gender + age group + social class
    # Master=young boy, Miss=unmarried female, Mr=adult male, Rare=Dr/Col/etc
    df = df.withColumn("Title", regexp_extract(col("Name"), r" ([A-Za-z]+)\.", 1))
    df = df.withColumn("Title",
        when(col("Title") == "Mr",                     "Mr")
       .when(col("Title").isin("Miss", "Mlle", "Ms"),  "Miss")
       .when(col("Title").isin("Mrs", "Mme"),           "Mrs")
       .when(col("Title") == "Master",                 "Master")
       .otherwise("Rare")
    )
    df = df.withColumn("family_size",     col("SibSp") + col("Parch") + lit(1))
    df = df.withColumn("is_alone",        when(col("family_size") == 1, 1).otherwise(0))
    df = df.withColumn("fare_per_person", col("Fare") / col("family_size"))
    df = df.withColumn("age_class",       col("Age")  * col("Pclass"))
    return df

df_eng = add_features(df)
df_eng.cache()

title_indexer = StringIndexer(inputCol="Title", outputCol="TitleIdx", handleInvalid="keep")
imputer_eng   = Imputer(
    inputCols  = ["Age", "Fare", "age_class", "fare_per_person"],
    outputCols = ["Age", "Fare", "age_class", "fare_per_person"],
    strategy   = "median"
)

ENG_FEATURES  = BASE_FEATURES + ["TitleIdx", "family_size", "is_alone", "fare_per_person", "age_class"]
assembler_eng = VectorAssembler(inputCols=ENG_FEATURES, outputCol="features", handleInvalid="keep")

rf_eng = RandomForestClassifier(
    labelCol="Survived", featuresCol="features",
    numTrees=100, maxDepth=5, minInstancesPerNode=2, seed=42
)

pipeline_eng = Pipeline(stages=[
    sex_indexer, embarked_indexer, title_indexer,
    imputer_eng, assembler_eng, rf_eng
])

eng_model, eng_auc, eng_train_acc, eng_train_f1 = run_cv(pipeline_eng, df_eng)

print(f"   New features : Title, family_size, is_alone, fare_per_person, age_class")
print(f"   CV AUC-ROC  (avg 5 folds) : {eng_auc:.4f}   (baseline: {base_auc:.4f})")
print(f"   Train Accuracy (full fit) : {eng_train_acc:.4f}")
print(f"   CV AUC Δ                  : {eng_auc - base_auc:+.4f}")

# ─────────────────────────────────────────────────────────────
# 6. IMPROVEMENT 2 — HYPERPARAMETER TUNING (REGULARIZED)
#    18 combos × 3 folds = 54 fits  (~3 minutes)
# ─────────────────────────────────────────────────────────────
print("\n[5/6] Improvement 2 — Hyperparameter Tuning  (3-fold CV)...")
print("   Testing regularized parameters to prevent overfitting...")

rf_tune = RandomForestClassifier(
    labelCol="Survived", featuresCol="features", seed=42
)

pipeline_tune = Pipeline(stages=[
    sex_indexer, embarked_indexer, title_indexer,
    imputer_eng, assembler_eng, rf_tune
])

# UPDATED: Constrained grid to force the model to generalize
param_grid = (
    ParamGridBuilder()
    .addGrid(rf_tune.numTrees,            [100, 200]) # Reduced max trees slightly to save time
    .addGrid(rf_tune.maxDepth,            [3, 4, 5])  # Lowered depth to prevent memorization
    .addGrid(rf_tune.minInstancesPerNode, [2, 5, 10]) # Forced grouping in leaves
    .build()
)

best_model, tune_auc, tune_train_acc, tune_train_f1 = run_cv(
    pipeline_tune, df_eng, param_grid, nfolds=3
)

best_rf       = best_model.stages[-1]
best_trees    = best_rf.getNumTrees
best_depth    = best_rf.getOrDefault("maxDepth")
best_min_inst = best_rf.getOrDefault("minInstancesPerNode")

print(f"   Best numTrees            : {best_trees}")
print(f"   Best maxDepth            : {best_depth}")
print(f"   Best minInstancesPerNode : {best_min_inst}")
print(f"   CV AUC-ROC  (avg 3 folds) : {tune_auc:.4f}   (baseline: {base_auc:.4f})")
print(f"   Train Accuracy (full fit) : {tune_train_acc:.4f}")
print(f"   CV AUC Δ                  : {tune_auc - base_auc:+.4f}")
# ─────────────────────────────────────────────────────────────
# 7. RESULTS SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

results = {
    "Baseline RF"          : {"CV AUC": base_auc, "Train Acc": base_train_acc, "Train F1": base_train_f1},
    "+Feature Engineering" : {"CV AUC": eng_auc,  "Train Acc": eng_train_acc,  "Train F1": eng_train_f1},
    "+Tuned Params"        : {"CV AUC": tune_auc, "Train Acc": tune_train_acc, "Train F1": tune_train_f1},
}

df_results = pd.DataFrame(results).T
print(df_results.to_string(float_format="{:.4f}".format))
print()
print("CV AUC  = mean AUC-ROC across folds  (primary, stable metric)")
print("Train * = scores on full 891-row dataset after final refit")

# ─────────────────────────────────────────────────────────────
# 8. SAVE RESULTS + CHART
# ─────────────────────────────────────────────────────────────
print("\n[6/6] Saving results...")

with open("results.json", "w") as f:
    json.dump(
        {k: {m: round(float(v), 4) for m, v in vals.items()}
         for k, vals in results.items()},
        f, indent=2
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
models  = list(results.keys())
colors  = ["#4a6cf7", "#f7874a", "#52c97a"]
cv_aucs = [results[m]["CV AUC"]   for m in models]
tr_accs = [results[m]["Train Acc"] for m in models]
tr_f1s  = [results[m]["Train F1"] for m in models]

# Left: CV AUC per model (primary metric)
bars = axes[0].bar(models, cv_aucs, color=colors, width=0.45)
axes[0].set_title("CV AUC-ROC  (primary metric)", fontsize=11)
axes[0].set_ylabel("AUC-ROC")
axes[0].set_ylim(0.80, 0.96)
axes[0].grid(axis="y", alpha=0.3)
for bar, v in zip(bars, cv_aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.002,
                 f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

# Right: Train Acc + Train F1
x = np.arange(len(models))
w = 0.3
axes[1].bar(x - w/2, tr_accs, w, label="Train Accuracy", color="#4a6cf7")
axes[1].bar(x + w/2, tr_f1s,  w, label="Train F1",       color="#f7874a")
axes[1].set_title("Train Accuracy & F1  (full dataset fit)", fontsize=11)
axes[1].set_ylabel("Score")
axes[1].set_ylim(0.75, 1.00)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, fontsize=9)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)
for b in axes[1].containers:
    axes[1].bar_label(b, fmt="%.3f", padding=2, fontsize=8)

plt.suptitle("Model Comparison — Titanic (PySpark MLlib)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)

print("✅ results.json saved")
print("✅ model_comparison.png saved")
print("\n🎉 Task 2 complete!")

spark.stop()

TASK 2: ML Basics & Optimization — PySpark + Titanic

✅ Spark session started  (version 4.0.2)

[1/6] Loading Titanic dataset...
   Rows       : 891
   Survived   : 342 (38.4%)
   Evaluation : 5-fold CV on full dataset (713 train / 178 test per fold)

[2/6] Preprocessing stages ready...

[3/6] Training Baseline Model  (5-fold CV)...
   CV AUC-ROC  (avg 5 folds) : 0.8688   ← primary metric
   Train Accuracy (full fit) : 0.8507
   Train F1      (full fit)  : 0.8475

[4/6] Improvement 1 — Feature Engineering  (5-fold CV)...
   New features : Title, family_size, is_alone, fare_per_person, age_class
   CV AUC-ROC  (avg 5 folds) : 0.8720   (baseline: 0.8688)
   Train Accuracy (full fit) : 0.8519
   CV AUC Δ                  : +0.0032

[5/6] Improvement 2 — Hyperparameter Tuning  (3-fold CV)...
   Testing regularized parameters to prevent overfitting...
   Best numTrees            : 200
   Best maxDepth            : 5
   Best minInstancesPerNode : 2
   CV AUC-ROC  (avg 3 folds) : 0.8724   (ba